### Feedback Analysis Agent using LangGraph

In [ ]:
! pip install langgraph langchain-google-genai pandas openpyxl pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 30.0 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.59.1 which is incompatible.


In [ ]:
import os
os.environ["GEMINI_API_KEY"] = "your_actual_gemini_api_key_here"
print("Key configured successfully:", "GEMINI_API_KEY" in os.environ)

Key configured successfully: True


In [ ]:
import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")


In [ ]:
import os
import pandas as pd
from typing import List, Optional
from pydantic import BaseModel, Field
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI

In [24]:
# 1. Define the Structured Output Schema
class FeedbackAnalysis(BaseModel):
    is_positive: bool = Field(
        description="True if the feedback column content is generally positive, False if negative"
    )
    strength_points: List[str] = Field(
        description="List of key strengths or praises mentioned in the feedback column"
    )
    weaknesses: List[str] = Field(
        description="List of weaknesses, complaints, or areas of improvement mentioned in the feedback column"
    )

In [25]:
# 2. Define LangGraph State
class GraphState(TypedDict):
    review_text: str
    analysis: Optional[FeedbackAnalysis]

In [26]:
# 3. Initialize Gemini and Force Schema Constraints
# Note: with_structured_output works natively with gemini-2.5 models
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
structured_llm = llm.with_structured_output(FeedbackAnalysis)

def analyze_feedback_node(state: GraphState) -> dict:
    prompt = f"Analyze the following user feedback thoroughly:\n\n{state['review_text']}"
    try:
        result = structured_llm.invoke(prompt)
        return {"analysis": result}
    except Exception as e:
        print(f"Error processing feedback item: {e}")
        return {"analysis": None}

In [27]:
# 4. Build and Compile the Graph Workflow
builder = StateGraph(GraphState)
builder.add_node("analyzer", analyze_feedback_node)
builder.add_edge(START, "analyzer")
builder.add_edge("analyzer", END)
graph = builder.compile()

In [28]:
# 5. Full Table Processing Function
def process_feedback_excel(input_file_path: str, output_file_path: str):
    print(f"Reading {input_file_path}...")
    df = pd.read_excel(input_file_path)

    # Validation check for the target text column
    if 'feedback' not in df.columns:
        raise ValueError("The Excel file must contain a column named exactly 'feedback'.")

    # Initialize response arrays matching the length of the dataframe
    is_positive_list = []
    strengths_list = []
    weaknesses_list = []

    total_rows = len(df)
    for idx, row in df.iterrows():
        # Keep everything else in the row intact; extract only the feedback column string value
        feedback_text = str(row['feedback']).strip()
        print(f"Processing record {idx + 1}/{total_rows}...")

        # Guard clause for empty rows or null entries
        if not feedback_text or feedback_text.lower() == 'nan':
            is_positive_list.append(None)
            strengths_list.append("")
            weaknesses_list.append("")
            continue

        # Execute the LangGraph Agent pipeline
        res = graph.invoke({"review_text": feedback_text})
        analysis = res.get("analysis")

        if analysis:
            is_positive_list.append(analysis.is_positive)
            strengths_list.append(", ".join(analysis.strength_points))
            weaknesses_list.append(", ".join(analysis.weaknesses))
        else:
            is_positive_list.append(None)
            strengths_list.append("Error processing sheet item")
            weaknesses_list.append("Error processing sheet item")

    # Append the results as new evaluation columns.
    # All other columns (name, email, product_name, rate, etc.) remain intact.
    df['feedback_label_positive'] = is_positive_list
    df['strength_points'] = strengths_list
    df['weaknesses'] = weaknesses_list

    # Export back out into a target spreadsheet layout
    df.to_excel(output_file_path, index=False)
    print(f"Workflow completed successfully! Output file: {output_file_path}")

In [29]:
if __name__ == "__main__":
    input_file = "customer_feedback.xlsx"
    output_file = "processed_feedback.xlsx"

    # Generate mock sheet detailing all specified columns if it doesn't exist
    if not os.path.exists(input_file):
        print(f"Generating full template '{input_file}' to match user specification...")
        sample_data = {
            "name": ["Alice Smith", "Bob Jones", "Charlie Brown"],
            "username": ["asmith99", "bjones_dev", "charlie_b"],
            "email": ["alice@email.com", "bob@email.com", "charlie@email.com"],
            "phone_number": ["555-0192", "555-0143", "555-0188"],
            "product_id": [1001, 1002, 1003],
            "product_name": ["Wireless Mouse", "Mechanical Keyboard", "USB-C Hub"],
            "rate": [9, 3, 10],
            "feedback": [
                "Ergonomics are wonderful and battery life is lasting weeks. A bit clicky though.",
                "Keys keep sticking during typing sessions and the backlight is dim. Worse value ever.",
                "Incredibly sturdy design, transfers data instantly, completely flawless!"
            ]
        }
        pd.DataFrame(sample_data).to_excel(input_file, index=False)

    process_feedback_excel(input_file, output_file)


Reading customer_feedback.xlsx...
Processing record 1/3...
Processing record 2/3...
Processing record 3/3...
Workflow completed successfully! Output file: processed_feedback.xlsx
